# Notebook 24 — Universality Collapse and Finite-Size Scaling

**Control-stack module:** universality collapse and finite-size scaling for phase-lock failure boundaries.

This notebook extends Notebook 23 by asking whether controller failures share reusable scaling structure. Instead of only fitting separate critical exponents, it normalizes margin curves, compares controller families, sweeps finite system sizes, and exports a collapse score for certification.

**Core idea:** if phase-lock failure is structured, then margins near failure should collapse under a shared scaling form:

```text
margin ≈ A · (s_crit − s)^β
```

where `s` is attack strength, `s_crit` is critical attack strength, `β` is an empirical critical-exponent proxy, and lower collapse error means stronger universality evidence.


## 1. Setup

Creates standard repo folders, constants, deterministic seed, and export paths.


In [ ]:
# ============================================================
# Setup + imports
# ============================================================

from pathlib import Path
import json
import math
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "24"
SLUG = "universality_collapse_and_finite_size_scaling"
TITLE = "Universality Collapse and Finite-Size Scaling"
SEED = 9423
THRESHOLD = 1 / np.sqrt(2)
MAX_ATTACK_STRENGTH = 5.0

np.random.seed(SEED)

ROOT = Path(".")
DOCS_DIR = ROOT / "docs"
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"
MATH_DIR = ROOT / "math"
for d in [DOCS_DIR, RESULTS_DIR, FIG_DIR, MATH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Notebook {NOTEBOOK_ID}: {TITLE}")
print(f"45° threshold = {THRESHOLD:.6f}")
print(f"outputs -> {FIG_DIR}, {RESULTS_DIR}, {DOCS_DIR}")


## 2. Synthetic phase-lock model

Uses the same policy names and phase-lock vocabulary as previous notebooks. The data model is intentionally lightweight, deterministic, and notebook-local so the file runs outside Colab or GitHub state.


In [ ]:
# ============================================================
# Helper functions and synthetic policy model
# ============================================================

POLICIES = [
    "none",
    "moving_average",
    "joint_kalman",
    "robust_gated_kalman",
    "constrained_mpc",
    "cgcs_invariance_control",
    "oracle",
]

FAMILY = {
    "none": "baseline",
    "moving_average": "average",
    "joint_kalman": "Kalman",
    "robust_gated_kalman": "Kalman",
    "constrained_mpc": "MPC",
    "cgcs_invariance_control": "CGCS",
    "oracle": "oracle",
}

BASE_CRIT = {
    "none": 1.20,
    "moving_average": 2.10,
    "robust_gated_kalman": 1.90,
    "joint_kalman": 2.30,
    "constrained_mpc": 2.30,
    "cgcs_invariance_control": 3.45,
    "oracle": 6.00,
}

BASE_BETA = {
    "none": 0.64,
    "moving_average": 0.63,
    "robust_gated_kalman": 0.48,
    "joint_kalman": 1.38,
    "constrained_mpc": 1.29,
    "cgcs_invariance_control": 0.69,
    "oracle": 0.30,
}

BASE_AMP = {
    "none": 0.18,
    "moving_average": 0.20,
    "robust_gated_kalman": 0.21,
    "joint_kalman": 0.23,
    "constrained_mpc": 0.24,
    "cgcs_invariance_control": 0.25,
    "oracle": 0.27,
}


def finite_size_factor(n_blocks: int) -> float:
    """Small finite-size correction: larger calibration horizons reduce boundary noise."""
    return 1.0 - 0.18 / np.sqrt(max(n_blocks, 1) / 48)


def policy_margin_curve(policy: str, strengths: np.ndarray, n_blocks: int, rng: np.random.Generator) -> np.ndarray:
    """Generate positive pre-boundary margins and negative post-boundary margins."""
    crit = BASE_CRIT[policy] * (1 + 0.035 * np.log2(n_blocks / 88))
    beta = BASE_BETA[policy]
    amp = BASE_AMP[policy] * finite_size_factor(n_blocks)
    distance = np.maximum(crit - strengths, 0)
    margin = amp * np.power(distance / max(crit, 1e-9), beta)

    # post-failure slope with deterministic family-specific roughness
    post = strengths > crit
    margin[post] = -0.07 * (strengths[post] - crit) ** 0.85

    ripple = 0.006 * np.sin(2.7 * strengths + 0.3 * len(policy))
    noise = rng.normal(0, 0.004 + 0.006 / np.sqrt(n_blocks / 48), size=len(strengths))
    if policy == "oracle":
        margin = np.maximum(0.18, margin + 0.02 * np.exp(-strengths / 5))
        noise *= 0.15
        ripple *= 0.15
    return margin + ripple + noise


def response_rmse(policy: str, strengths: np.ndarray, n_blocks: int, rng: np.random.Generator) -> np.ndarray:
    """Synthetic response RMSE increases with attack strength but improves with better policies."""
    base = {
        "oracle": 0.0,
        "cgcs_invariance_control": 0.006,
        "constrained_mpc": 0.008,
        "joint_kalman": 0.010,
        "moving_average": 0.012,
        "robust_gated_kalman": 0.013,
        "none": 0.016,
    }[policy]
    slope = {
        "oracle": 0.0,
        "cgcs_invariance_control": 0.0015,
        "constrained_mpc": 0.0022,
        "joint_kalman": 0.0027,
        "moving_average": 0.0034,
        "robust_gated_kalman": 0.0037,
        "none": 0.0044,
    }[policy]
    scale = np.sqrt(88 / n_blocks)
    y = base + slope * strengths * scale + rng.normal(0, 0.0008 * scale, size=len(strengths))
    return np.clip(y, 0, None)


def first_failure_strength(strengths: np.ndarray, margins: np.ndarray) -> float:
    idx = np.where(margins < 0)[0]
    if len(idx) == 0:
        return float(strengths[-1])
    return float(strengths[idx[0]])


def fit_scaling(strengths: np.ndarray, margins: np.ndarray, s_crit: float, min_points: int = 5):
    """Fit log(margin)=log(A)+β log(scrit-s) over near-boundary positive points."""
    mask = (margins > 0) & (strengths < s_crit)
    x = s_crit - strengths[mask]
    y = margins[mask]
    if len(x) < min_points:
        return np.nan, np.nan, np.nan
    # keep closest 45% of pre-failure samples, with enough points
    cutoff = np.quantile(x, 0.45)
    near = x <= cutoff
    if near.sum() < min_points:
        near = np.argsort(x)[:min_points]
    lx = np.log(np.maximum(x[near], 1e-8))
    ly = np.log(np.maximum(y[near], 1e-8))
    beta, logA = np.polyfit(lx, ly, 1)
    pred = logA + beta * lx
    rmse = float(np.sqrt(np.mean((ly - pred) ** 2)))
    return float(beta), float(np.exp(logA)), rmse


def savefig(name: str):
    path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"saved figure: {path}")
    return path


## 3. Generate finite-size sweeps

Build a compact certification dataset for multiple calibration-block horizons. This gives Notebook 24 a finite-size axis rather than only one fixed run length.


In [ ]:
# ============================================================
# Generate finite-size sweep data
# ============================================================

strengths = np.round(np.linspace(0, MAX_ATTACK_STRENGTH, 51), 3)
block_sizes = [48, 64, 88, 128]
rows = []

for n_blocks in block_sizes:
    rng = np.random.default_rng(SEED + n_blocks)
    for policy in POLICIES:
        margins = policy_margin_curve(policy, strengths, n_blocks, rng)
        rmses = response_rmse(policy, strengths, n_blocks, rng)
        s_fail = first_failure_strength(strengths, margins)
        beta, amp, fit_rmse = fit_scaling(strengths, margins, s_fail)
        for s, margin, rmse in zip(strengths, margins, rmses):
            rows.append({
                "n_blocks": n_blocks,
                "policy": policy,
                "family": FAMILY[policy],
                "attack_strength": float(s),
                "margin": float(margin),
                "response_rmse": float(rmse),
                "first_failure_strength": s_fail,
                "beta_fit": beta,
                "amplitude_fit": amp,
                "fit_log_rmse": fit_rmse,
            })

sweep_df = pd.DataFrame(rows)
sweep_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_finite_size_sweep.csv"
sweep_df.to_csv(sweep_path, index=False)
print(f"saved sweep: {sweep_path}")
sweep_df.head()


## 4. Critical summaries by policy and size

Collapse-ready table: critical strength, empirical β, fitted amplitude, fitting error, and RMSE by policy and calibration horizon.


In [ ]:
# ============================================================
# Build summary tables
# ============================================================

summary_rows = []
for (n_blocks, policy), g in sweep_df.groupby(["n_blocks", "policy"]):
    g = g.sort_values("attack_strength")
    margin = g["margin"].to_numpy()
    s = g["attack_strength"].to_numpy()
    s_fail = first_failure_strength(s, margin)
    beta, amp, fit_rmse = fit_scaling(s, margin, s_fail)
    summary_rows.append({
        "n_blocks": n_blocks,
        "policy": policy,
        "family": FAMILY[policy],
        "critical_strength": s_fail,
        "beta_fit": beta,
        "amplitude_fit": amp,
        "fit_log_rmse": fit_rmse,
        "mean_rmse": g["response_rmse"].mean(),
        "p95_rmse": g["response_rmse"].quantile(0.95),
        "min_margin": g["margin"].min(),
        "blocks_below_threshold": int((g["margin"] < 0).sum()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["n_blocks", "critical_strength"], ascending=[True, False])
summary_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"saved summary: {summary_path}")
summary_df.head(12)


## 5. Normalized margin collapse

Plot margin collapse for the main system size. The x-axis is distance to first failure, and the y-axis is amplitude-normalized margin.


In [ ]:
# ============================================================
# Figure 1: normalized margin collapse
# ============================================================

main_n = 88
plt.figure(figsize=(12, 7))
for policy in POLICIES:
    g = sweep_df[(sweep_df["n_blocks"] == main_n) & (sweep_df["policy"] == policy)].sort_values("attack_strength")
    row = summary_df[(summary_df["n_blocks"] == main_n) & (summary_df["policy"] == policy)].iloc[0]
    s_crit = row["critical_strength"]
    amp = row["amplitude_fit"] if np.isfinite(row["amplitude_fit"]) and row["amplitude_fit"] > 0 else 1.0
    mask = (g["attack_strength"] < s_crit) & (g["margin"] > 0)
    x = s_crit - g.loc[mask, "attack_strength"].to_numpy()
    y = g.loc[mask, "margin"].to_numpy() / amp
    order = np.argsort(x)
    plt.plot(x[order], y[order], marker="o", label=f"{policy}")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("distance to critical strength: s_crit - s")
plt.ylabel("amplitude-normalized margin")
plt.title("Universality collapse: normalized pre-failure margins")
plt.legend(ncol=2)
savefig("normalized_margin_collapse")


## 6. Log-log collapse by policy

Same data with explicit fitted slopes. Parallel-ish lines support a shared family structure; separated slopes suggest distinct controller universality classes.


In [ ]:
# ============================================================
# Figure 2: log-log collapse by policy with fitted slopes
# ============================================================

plt.figure(figsize=(12, 7))
for policy in POLICIES:
    if policy == "oracle":
        continue
    g = sweep_df[(sweep_df["n_blocks"] == main_n) & (sweep_df["policy"] == policy)].sort_values("attack_strength")
    row = summary_df[(summary_df["n_blocks"] == main_n) & (summary_df["policy"] == policy)].iloc[0]
    s_crit, beta, amp = row["critical_strength"], row["beta_fit"], row["amplitude_fit"]
    mask = (g["attack_strength"] < s_crit) & (g["margin"] > 0)
    x = s_crit - g.loc[mask, "attack_strength"].to_numpy()
    y = g.loc[mask, "margin"].to_numpy()
    near = x <= np.quantile(x, 0.6)
    plt.scatter(np.log(x[near]), np.log(y[near]), label=f"{policy} β={beta:.2f}")
    xs = np.linspace(x[near].min(), x[near].max(), 60)
    plt.plot(np.log(xs), np.log(np.maximum(amp, 1e-8)) + beta * np.log(xs), alpha=0.8)
plt.axhline(0, linestyle="--", alpha=0.4)
plt.xlabel("log(s_crit - s)")
plt.ylabel("log(margin)")
plt.title("Universality collapse: log-log pre-failure scaling")
plt.legend(ncol=2)
savefig("loglog_collapse_by_policy")


## 7. Exponent flatness check

If `margin / (s_crit - s)^β` is stable near failure, then fitted scaling is locally coherent.


In [ ]:
# ============================================================
# Figure 3: exponent flatness check
# ============================================================

plt.figure(figsize=(12, 7))
for policy in POLICIES:
    if policy == "oracle":
        continue
    g = sweep_df[(sweep_df["n_blocks"] == main_n) & (sweep_df["policy"] == policy)].sort_values("attack_strength")
    row = summary_df[(summary_df["n_blocks"] == main_n) & (summary_df["policy"] == policy)].iloc[0]
    s_crit, beta = row["critical_strength"], row["beta_fit"]
    mask = (g["attack_strength"] < s_crit) & (g["margin"] > 0)
    x = s_crit - g.loc[mask, "attack_strength"].to_numpy()
    y = g.loc[mask, "margin"].to_numpy() / np.power(np.maximum(x, 1e-8), beta)
    order = np.argsort(x)
    plt.plot(x[order], y[order], marker="o", label=policy)
plt.xscale("log")
plt.xlabel("distance to critical strength: s_crit - s")
plt.ylabel("margin / (s_crit - s)^β")
plt.title("Exponent flatness check near phase-lock boundary")
plt.legend(ncol=2)
savefig("exponent_flatness_check")


## 8. Family-level collapse overlay

Collapse by controller family, not individual policy. This asks whether CGCS, MPC, Kalman, average, and baseline form distinguishable classes.


In [ ]:
# ============================================================
# Figure 4: family collapse overlay
# ============================================================

plt.figure(figsize=(12, 7))
for fam, fam_df in sweep_df[sweep_df["n_blocks"] == main_n].groupby("family"):
    xs_all, ys_all = [], []
    for policy, g in fam_df.groupby("policy"):
        row = summary_df[(summary_df["n_blocks"] == main_n) & (summary_df["policy"] == policy)].iloc[0]
        s_crit = row["critical_strength"]
        amp = row["amplitude_fit"] if np.isfinite(row["amplitude_fit"]) and row["amplitude_fit"] > 0 else 1.0
        mask = (g["attack_strength"] < s_crit) & (g["margin"] > 0)
        xs_all.extend((s_crit - g.loc[mask, "attack_strength"]).tolist())
        ys_all.extend((g.loc[mask, "margin"] / amp).tolist())
    xs_all, ys_all = np.array(xs_all), np.array(ys_all)
    order = np.argsort(xs_all)
    plt.plot(xs_all[order], ys_all[order], marker="o", linestyle="", label=fam, alpha=0.8)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("family-normalized distance to failure")
plt.ylabel("amplitude-normalized margin")
plt.title("Family-level collapse overlay")
plt.legend()
savefig("family_collapse_overlay")


## 9. Collapse score by family

Define collapse error as binned variance of normalized log-margin. Lower values mean curves inside a family collapse more cleanly.


In [ ]:
# ============================================================
# Collapse error by family
# ============================================================

def collapse_error_for_family(fam: str, n_blocks: int = main_n, bins: int = 8) -> float:
    rows = []
    fam_df = sweep_df[(sweep_df["n_blocks"] == n_blocks) & (sweep_df["family"] == fam)]
    for policy, g in fam_df.groupby("policy"):
        row = summary_df[(summary_df["n_blocks"] == n_blocks) & (summary_df["policy"] == policy)].iloc[0]
        s_crit = row["critical_strength"]
        amp = row["amplitude_fit"] if np.isfinite(row["amplitude_fit"]) and row["amplitude_fit"] > 0 else 1.0
        mask = (g["attack_strength"] < s_crit) & (g["margin"] > 0)
        x = s_crit - g.loc[mask, "attack_strength"].to_numpy()
        y = g.loc[mask, "margin"].to_numpy() / amp
        for xi, yi in zip(x, y):
            rows.append({"policy": policy, "logx": np.log(max(xi, 1e-8)), "logy": np.log(max(yi, 1e-8))})
    df = pd.DataFrame(rows)
    if len(df) < bins or df["policy"].nunique() < 2:
        return 0.0
    df["bin"] = pd.qcut(df["logx"], q=min(bins, len(df)//2), duplicates="drop")
    variances = df.groupby("bin", observed=True)["logy"].var().dropna()
    return float(variances.mean()) if len(variances) else 0.0

collapse_rows = []
for fam in sorted(set(FAMILY.values())):
    collapse_rows.append({"family": fam, "collapse_error": collapse_error_for_family(fam)})
collapse_df = pd.DataFrame(collapse_rows).sort_values("collapse_error")
collapse_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_collapse_scores.csv"
collapse_df.to_csv(collapse_path, index=False)
print(f"saved collapse scores: {collapse_path}")
collapse_df


In [ ]:
# ============================================================
# Figure 5: collapse error by family
# ============================================================

plt.figure(figsize=(10, 6))
plt.bar(collapse_df["family"], collapse_df["collapse_error"])
plt.xlabel("controller family")
plt.ylabel("collapse error (lower = cleaner collapse)")
plt.title("Collapse error by controller family")
savefig("collapse_error_by_family")


## 10. Finite-size scaling

Repeat summary plots across calibration horizons. A robust controller should preserve critical strength and avoid explosive β drift as the horizon changes.


In [ ]:
# ============================================================
# Figure 6: critical strength vs system size
# ============================================================

plt.figure(figsize=(12, 7))
for policy in POLICIES:
    g = summary_df[summary_df["policy"] == policy].sort_values("n_blocks")
    plt.plot(g["n_blocks"], g["critical_strength"], marker="o", label=policy)
plt.xlabel("calibration horizon / blocks")
plt.ylabel("critical attack strength")
plt.title("Finite-size scaling: critical strength vs system size")
plt.legend(ncol=2)
savefig("critical_strength_vs_system_size")


In [ ]:
# ============================================================
# Figure 7: beta vs system size
# ============================================================

plt.figure(figsize=(12, 7))
for policy in POLICIES:
    if policy == "oracle":
        continue
    g = summary_df[summary_df["policy"] == policy].sort_values("n_blocks")
    plt.plot(g["n_blocks"], g["beta_fit"], marker="o", label=policy)
plt.xlabel("calibration horizon / blocks")
plt.ylabel("empirical β estimate")
plt.title("Finite-size scaling: β estimate vs system size")
plt.legend(ncol=2)
savefig("beta_vs_system_size")


In [ ]:
# ============================================================
# Figure 8: finite-size phase diagram
# ============================================================

pivot = summary_df.pivot(index="policy", columns="n_blocks", values="critical_strength").loc[POLICIES]
plt.figure(figsize=(10, 7))
plt.imshow(pivot, aspect="auto")
plt.colorbar(label="critical strength")
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.xlabel("calibration horizon / blocks")
plt.ylabel("policy")
plt.title("Finite-size phase diagram: critical strength")
savefig("finite_size_phase_diagram")


In [ ]:
# ============================================================
# Figure 9: RMSE vs system size
# ============================================================

plt.figure(figsize=(12, 7))
for policy in POLICIES:
    g = summary_df[summary_df["policy"] == policy].sort_values("n_blocks")
    plt.plot(g["n_blocks"], g["mean_rmse"], marker="o", label=policy)
plt.xlabel("calibration horizon / blocks")
plt.ylabel("mean response RMSE")
plt.title("Finite-size scaling: RMSE vs system size")
plt.legend(ncol=2)
savefig("rmse_vs_system_size")


## 11. Universality summary table

Combines critical strength, β, fit error, collapse error, and RMSE into one compact table.


In [ ]:
# ============================================================
# Figure 10 + table: universality summary
# ============================================================

main_summary = summary_df[summary_df["n_blocks"] == main_n].copy()
main_summary = main_summary.merge(collapse_df, on="family", how="left")
main_summary["universality_score"] = (
    0.45 * main_summary["critical_strength"]
    - 0.20 * main_summary["mean_rmse"] * 100
    - 0.20 * main_summary["fit_log_rmse"].fillna(0)
    - 0.15 * main_summary["collapse_error"].fillna(0)
)
main_summary = main_summary.sort_values("universality_score", ascending=False)
summary_table_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_universality_summary.csv"
main_summary.to_csv(summary_table_path, index=False)
print(f"saved universality summary: {summary_table_path}")

fig, ax = plt.subplots(figsize=(15, 4.5))
ax.axis("off")
cols = ["policy", "family", "critical_strength", "beta_fit", "fit_log_rmse", "collapse_error", "mean_rmse", "universality_score"]
display_df = main_summary[cols].copy()
for c in ["critical_strength", "beta_fit", "fit_log_rmse", "collapse_error", "mean_rmse", "universality_score"]:
    display_df[c] = display_df[c].map(lambda x: f"{x:.3f}" if np.isfinite(x) else "n/a")
table = ax.table(cellText=display_df.values, colLabels=display_df.columns, loc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)
ax.set_title("Universality summary table", pad=20)
savefig("universality_summary_table")

main_summary[cols]


## 12. README snippet, markdown report, manifest, and ZIP export

Writes reproducible repo artifacts using the same structure as earlier notebooks.


In [ ]:
# ============================================================
# Write README snippet, markdown report, manifest, and zip export
# ============================================================

figure_names = [
    "normalized_margin_collapse",
    "loglog_collapse_by_policy",
    "exponent_flatness_check",
    "family_collapse_overlay",
    "collapse_error_by_family",
    "critical_strength_vs_system_size",
    "beta_vs_system_size",
    "finite_size_phase_diagram",
    "rmse_vs_system_size",
    "universality_summary_table",
]

readme_snippet = f"""## Notebook {NOTEBOOK_ID}: {TITLE}

Notebook 24 tests whether controller failure curves collapse into reusable universality classes.
It normalizes phase-lock margins near critical attack strength, estimates family-level collapse error,
and repeats the analysis across finite calibration horizons.

Primary outputs:

- `results/{NOTEBOOK_ID}_{SLUG}_finite_size_sweep.csv`
- `results/{NOTEBOOK_ID}_{SLUG}_summary.csv`
- `results/{NOTEBOOK_ID}_{SLUG}_collapse_scores.csv`
- `results/{NOTEBOOK_ID}_{SLUG}_universality_summary.csv`
- `docs/{NOTEBOOK_ID}_{SLUG}.md`
"""

snippet_path = DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}_README_snippet.md"
snippet_path.write_text(readme_snippet)
print(f"saved README snippet: {snippet_path}")

best_policy = main_summary.iloc[0]["policy"]
best_family = main_summary.iloc[0]["family"]
best_score = main_summary.iloc[0]["universality_score"]

md_lines = []
md_lines.append(f"# Notebook {NOTEBOOK_ID} — {TITLE}")
md_lines.append("")
md_lines.append("## Summary")
md_lines.append("")
md_lines.append(
    "Notebook 24 converts phase-transition boundary curves into a universality-collapse test. "
    "It compares amplitude-normalized margins, fitted critical-exponent proxies, collapse error by family, "
    "and finite-size behavior across calibration horizons."
)
md_lines.append("")
md_lines.append("## Main result")
md_lines.append("")
md_lines.append(f"- Best policy by universality score: `{best_policy}`")
md_lines.append(f"- Best controller family: `{best_family}`")
md_lines.append(f"- Best universality score: `{best_score:.3f}`")
md_lines.append("")
md_lines.append("## Interpretation")
md_lines.append("")
md_lines.append(
    "The collapse analysis treats failure onset as a phase-lock boundary rather than a one-off error event. "
    "Policies with higher critical strength and lower collapse error behave like more stable universality classes. "
    "Finite-size sweeps test whether that structure persists as calibration horizon changes."
)
md_lines.append("")
md_lines.append("## Figures")
md_lines.append("")
for name in figure_names:
    md_lines.append(f"![{name}](../figures/{NOTEBOOK_ID}_{SLUG}_{name}.png)")
md_lines.append("")
md_lines.append("## Universality summary")
md_lines.append("")
md_lines.append(main_summary[cols].to_markdown(index=False))
md_lines.append("")

md_path = DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}.md"
md_path.write_text("\n".join(md_lines))
print(f"saved markdown: {md_path}")

manifest = {
    "notebook": f"{NOTEBOOK_ID}_{SLUG}",
    "title": TITLE,
    "threshold": float(THRESHOLD),
    "seed": SEED,
    "block_sizes": block_sizes,
    "max_attack_strength": MAX_ATTACK_STRENGTH,
    "best_policy": str(best_policy),
    "best_family": str(best_family),
    "figures": [str(FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png") for name in figure_names],
    "results": [
        str(sweep_path),
        str(summary_path),
        str(collapse_path),
        str(summary_table_path),
    ],
    "docs": [str(md_path), str(snippet_path)],
}
manifest_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"saved manifest: {manifest_path}")

zip_path = ROOT / f"{NOTEBOOK_ID}_{SLUG}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in manifest["figures"] + manifest["results"] + manifest["docs"] + [str(manifest_path)]:
        p = Path(path)
        if p.exists():
            zf.write(p, arcname=str(p))
print(f"saved zip: {zip_path}")


## 13. Optional Colab download

Run this cell in Colab to download the output bundle.


In [ ]:
# ============================================================
# Optional Colab download
# ============================================================

try:
    from google.colab import files
    files.download(f"{NOTEBOOK_ID}_{SLUG}_outputs.zip")
except Exception as exc:
    print("Download skipped outside Colab or before zip creation.")
    print(exc)
